# Пеленг: воспроизводимое сравнение геометрии

**Эксперимент v4 не принят для финальной сдачи.** На Way1 RMSE снизилась, но доля кадров в 5 м ухудшилась; на scene2 покрытие центра упало с 93.01% до 47.41%. Этот ноутбук воспроизводит эксперимент, а не запускает подтверждённо лучший вариант. [Итоговый отчёт](https://github.com/petrussia/3d-track-detection-and-tracking/blob/main/docs/geometry-benchmark.md).

Все вычисления выполняются на GPU Colab. Данные и веса ожидаются в `MyDrive/Peleng-Colab`, подготовленном для проекта. Новое обучение не запускается.

Сравниваем исходный код, исправления и исправления с заданными габаритами на **Way1**. Детектор один и тот же. Затем восстанавливаем центр грузовика на **scene2** с адаптированной YOLO11s.

Критерии: RMSE по XYZ и 3D на одинаковых кадрах, покрытие и доля всех валидных кадров эталона в допусках 1/2/5/10/15/20 м. Пропуски не считаются успехом. Для Way1 используется центр передней грани: это визуально обоснованное приближение, точный pivot организаторов не подтверждён. Для scene2 эталона нет — её точность в метрах неизвестна.

Все четыре прогона запускаются заново в одном окружении. CSV эталона читается только после обработки видео. Результаты сохраняются в `MyDrive/Peleng-Colab/geometry-benchmark-reproduction`.


In [ ]:
# @title 1. Установка и проверка GPU
from pathlib import Path
import sys, subprocess
SOURCE=Path('/content/peleng')
if not SOURCE.exists():
    subprocess.run(['git','clone','https://github.com/petrussia/3d-track-detection-and-tracking.git',str(SOURCE)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(SOURCE/'requirements-colab.txt'),'ultralytics==8.4.202'],check=True)
import torch
assert torch.cuda.is_available(), 'Выберите GPU T4 в настройках среды Colab'
print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
# @title 2. Данные, фиксированная версия кода и проверка весов
from pathlib import Path
import sys, subprocess, json, hashlib, signal, shutil
import torch, ultralytics
from google.colab import drive
drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Нужна среда с GPU'
ROOT=Path('/content/drive/MyDrive/Peleng-Colab')
SOURCE=Path('/content/peleng')
CODE=Path('/content/peleng-geometry-v2')
COMMIT='3c9f8699ec6599f1b4b4ad3f5db69d093ae6dc7c'
subprocess.run(['git','-C',str(SOURCE),'fetch','origin','main'],check=True)
if not CODE.exists(): subprocess.run(['git','-C',str(SOURCE),'worktree','add','--detach',str(CODE),COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(CODE),'rev-parse','HEAD'],text=True).strip()==COMMIT
if str(CODE) not in sys.path: sys.path.insert(0,str(CODE))
from peleng.colab_support import unpack_bundles
items=[x for x in json.loads((ROOT/'bundle-manifest.json').read_text()) if x['name']=='organizer']
stage=Path('/content/organizer-bundle-only'); stage.mkdir(exist_ok=True)
(stage/'bundle-manifest.json').write_text(json.dumps(items))
for part in items[0]['parts']:
    link=stage/part
    if not link.exists(): link.symlink_to(ROOT/part)
unpack_bundles(stage,'/content/peleng-data')
unpack_bundles(ROOT/'scene2-input','/content/peleng-scene2')
BASE=Path('/content/peleng-baseline-23decea')
if not BASE.exists(): subprocess.run(['git','-C',str(SOURCE),'worktree','add','--detach',str(BASE),'23decea3ae146e2f2a08b2fcad66bf39870d5c51'],check=True)
base_weights=ROOT/'yolo11s.pt'
scene_weights=ROOT/'scene2-adaptation-v1/detector/weights/last.pt'
assert hashlib.sha256(base_weights.read_bytes()).hexdigest()=='85a76fe86dd8afe384648546b56a7a78580c7cb7b404fc595f97969322d502d5'
assert hashlib.sha256(scene_weights.read_bytes()).hexdigest()=='b996bed291c04e6609b4bc2a0bfc0d18a1352ae32fdbe14d3c7bd93c78a69fee'


In [ ]:
# @title 3. Четыре полных прогона и сравнение
OUT=Path('/content/geometry-benchmark-reproduction'); BACKUP=ROOT/'geometry-benchmark-reproduction'
OUT.mkdir(exist_ok=True); BACKUP.mkdir(exist_ok=True)
variants=['way1-baseline','way1-generic','way1-known','scene2-generic']
(BACKUP/'pip-freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print('GPU:',torch.cuda.get_device_name(0),'YOLO:',ultralytics.__version__,'commit:',COMMIT,flush=True)
command=[sys.executable,'-u',str(CODE/'scripts/run_geometry_benchmark.py'),
 '--way1','/content/peleng-data/datasets/organizer','--scene2','/content/peleng-scene2/datasets/scene2',
 '--way1-weights',str(base_weights),'--scene2-weights',str(scene_weights),
 '--baseline-code',str(BASE),'--outputs',str(OUT),'--backup',str(BACKUP)]
with (BACKUP/'progress.log').open('a') as log:
    for variant in variants:
        process=subprocess.Popen(command+['--only',variant],cwd=CODE,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='',flush=True); log.write(line); log.flush()
            if process.wait(): raise RuntimeError('Ошибка прогона: '+variant)
        except KeyboardInterrupt:
            process.send_signal(signal.SIGINT); process.wait(); raise
subprocess.run([sys.executable,str(CODE/'scripts/compare_geometry_benchmark.py'),str(OUT),
 '--truth','/content/peleng-data/datasets/organizer/object_transform_SM_AA_track.csv',
 '--output',str(BACKUP/'comparison.json')],check=True,stdout=subprocess.DEVNULL)
subprocess.run([sys.executable,str(CODE/'scripts/plot_geometry_benchmark.py'),str(OUT),
 '--truth','/content/peleng-data/datasets/organizer/object_transform_SM_AA_track.csv',
 '--output',str(BACKUP/'way1-comparison.png')],check=True)
print('Готово:',BACKUP,flush=True)

result=json.loads((BACKUP/'comparison.json').read_text())
print('Общие кадры Way1:', result['common_frames'])
for name, value in result['variants'].items():
    metric=value['common_frame_metrics']
    within5=next(x['pct_of_all_valid_truth'] for x in value['thresholds'] if x['tolerance_m']==5)
    print(name, 'RMSE 3D:', round(metric['three_d']['rmse_m'],3), 'м;',
          'покрытие:', round(value['named_reference_coverage_pct'],2), '%;',
          'до 5 м от всех корректных кадров:', round(within5,2), '%')
from IPython.display import display,Image
display(Image(filename=str(BACKUP/'way1-comparison.png')))


## Интерпретация

Смотрите `comparison.json` и `way1-comparison.png`. Меньшая ошибка при меньшем покрытии не является безусловным улучшением. Для scene2 смотрите `diagnostics.json`; покрытие и скачки не заменяют измерение точности. Фиксированные габариты остаются экспериментальным режимом.
